# 🎵 Zero-Shot Audio Processing (Colab Interactive Edition)

ออกแบบใหม่โดยแยกเป็น 3 ขั้นตอนย่อย และตัดไลบรารี Web UI ออก เพื่อรองรับการใช้งานแบบ Interactive บน Google Colab

In [ ]:
#@title 1. ติดตั้งไลบรารีและเตรียมโมเดล (รันแค่ครั้งแรกครั้งเดียว)
REPO_URL = "https://github.com/GiantAILab/YingMusic-SVC" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

import os, shutil
from IPython import get_ipython

ip = get_ipython()
WORK_DIR = "/content/audio_project"
READY_FLAG = "/content/.env_ready"

if not os.path.exists(READY_FLAG):
    print("📥 กำลังเตรียมไฟล์โปรเจกต์...")
    if not os.path.exists(WORK_DIR):
        ip.system(f"git clone -q --depth 1 {REPO_URL} {WORK_DIR}")
    os.chdir(WORK_DIR)

    print("📦 กำลังติดตั้งไลบรารีเสียง (ตัดโมดูล Web UI ออกแล้ว)...")
    ip.system("apt-get -qq update && apt-get -qq install -y sox libsox-fmt-all ffmpeg > /dev/null")
    ip.system(
        'pip install -q "librosa>=0.10.2,<1" "transformers>=4.46.3,<5" '
        '"huggingface-hub>=0.28.1,<1.0" munch einops "descript-audio-codec==1.0.0" '
        'pydub soundfile pyyaml omegaconf ml_collections loralib beartype '
        '"rotary_embedding_torch==0.3.5" matplotlib scipy tqdm audiomentations pedalboard'
    )

    print("⬇️ กำลังดาวน์โหลดโมเดลจาก Hugging Face...")
    from huggingface_hub import hf_hub_download, snapshot_download
    token = HF_TOKEN.strip() or None

    os.makedirs("checkpoints", exist_ok=True)
    os.makedirs("accom_separation/ckpt/bs_roformer", exist_ok=True)

    hf_hub_download("GiantAILab/YingMusic-SVC", "YingMusic-SVC-full.pt", local_dir="checkpoints", token=token)
    hf_hub_download("GiantAILab/YingMusic-SVC", "bs_roformer.ckpt", local_dir="accom_separation/ckpt/bs_roformer", token=token)

    rmvpe_path = hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", cache_dir="./checkpoints", token=token)
    camp_path = hf_hub_download("funasr/campplus", "campplus_cn_common.bin", cache_dir="./checkpoints", token=token)
    if not os.path.exists("checkpoints/rmvpe.pt"):
        shutil.copy(rmvpe_path, "checkpoints/rmvpe.pt")
    if not os.path.exists("checkpoints/campplus_cn_common.bin"):
        shutil.copy(camp_path, "checkpoints/campplus_cn_common.bin")

    snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", cache_dir="./checkpoints/hf_cache", allow_patterns=["config.json", "bigvgan_generator.pt"], token=token)
    snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", local_dir="./nvidia/bigvgan_v2_44khz_128band_512x", allow_patterns=["config.json", "bigvgan_generator.pt"], token=token)
    snapshot_download("openai/whisper-small", cache_dir="./checkpoints/hf_cache", allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"], token=token)
    snapshot_download("openai/whisper-small", local_dir="./openai/whisper-small", allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"], token=token)

    open(READY_FLAG, "w").close()
    print("✅ ติดตั้งเสร็จสมบูรณ์! กดรันข้อ 2 ต่อได้เลยครับ")
else:
    os.chdir(WORK_DIR)
    print("✅ ระบบพร้อมใช้งานอยู่แล้ว กดรันข้อ 2 ได้เลยครับ")

In [ ]:
#@title 2. อัปโหลดไฟล์และแยกเสียงดนตรี
TIMBRE = "upload" #@param ["upload", "prompts/tp.wav", "prompts/syz.wav"]
HAS_ACCOMPANIMENT = True #@param {type:"boolean"}
KEEP_BACKING_VOCAL = False #@param {type:"boolean"}

import os, shutil, time, shlex
from google.colab import files
from IPython import get_ipython

ip = get_ipython()
q = shlex.quote
os.chdir("/content/audio_project")

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
WORK = os.path.abspath(f"runs/{RUN_ID}")
os.makedirs(WORK, exist_ok=True)

def upload_file(msg):
    print(f"\n{msg}")
    up = files.upload()
    if not up:
        raise SystemExit("❌ ไม่ได้อัปโหลดไฟล์")
    fname = next(iter(up))
    safe = os.path.basename(fname).replace(" ", "_")
    dst = os.path.join(WORK, "raw_" + safe)
    shutil.move(fname, dst)
    return dst, os.path.splitext(safe)[0]

song_raw, SONG_TITLE = upload_file("🎵 อัปโหลดเพลงต้นฉบับ (mp3 / wav / flac / m4a):")
SONG_WAV = os.path.join(WORK, "song.wav")
ip.system(f"ffmpeg -hide_banner -loglevel error -y -i {q(song_raw)} -ar 44100 {q(SONG_WAV)}")

if TIMBRE == "upload":
    timbre_raw, _ = upload_file("🗣️ อัปโหลดเสียงเป้าหมาย (เสียงร้องหรือพูดสะอาด 10–25 วินาที):")
    TIMBRE_WAV = os.path.join(WORK, "timbre.wav")
    ip.system(f"ffmpeg -hide_banner -loglevel error -y -i {q(timbre_raw)} -ar 44100 -ac 1 {q(TIMBRE_WAV)}")
else:
    TIMBRE_WAV = os.path.abspath(TIMBRE)

if HAS_ACCOMPANIMENT:
    print("\n🎚️ กำลังแยกเสียงร้องและดนตรีประกอบ...")
    sep_in = os.path.join(WORK, "sep_in")
    sep_out = os.path.join(WORK, "sep_out")
    os.makedirs(sep_in, exist_ok=True)
    shutil.copy(SONG_WAV, sep_in)

    sep_cfg = os.path.abspath("accom_separation/ckpt/bs_roformer/config_bd_roformer.yaml")
    sep_ckpt = os.path.abspath("accom_separation/ckpt/bs_roformer/bs_roformer.ckpt")

    os.chdir("/content/audio_project/accom_separation")
    ip.system(
        f"PYTHONWARNINGS=ignore python inference.py --model_type bs_roformer "
        f"--config_path {q(sep_cfg)} --start_check_point {q(sep_ckpt)} "
        f"--input_folder {q(sep_in)} --store_dir {q(sep_out)} --disable_detailed_pbar"
    )
    os.chdir("/content/audio_project")

    stems = os.path.join(sep_out, "song")
    SOURCE_VOCAL = os.path.join(stems, "vocals.wav")
    ACCOMPANY = os.path.join(stems, "instrumental.wav")

    if KEEP_BACKING_VOCAL:
        backing = os.path.join(stems, "backing_vocal.wav")
        if os.path.exists(backing):
            mixed = os.path.join(stems, "instrumental_backing.wav")
            ip.system(f"ffmpeg -hide_banner -loglevel error -y -i {q(ACCOMPANY)} -i {q(backing)} -filter_complex amix=inputs=2:normalize=0 {q(mixed)}")
            ACCOMPANY = mixed
else:
    SOURCE_VOCAL = SONG_WAV
    ACCOMPANY = ""

print("✅ เตรียมไฟล์เสียงเสร็จแล้ว! กดรันข้อ 3 เพื่อเริ่มแปลงเสียงได้เลยครับ")

In [ ]:
#@title 3. แปลงเสียงร้องและดาวน์โหลดผลลัพธ์
DIFFUSION_STEPS = 100 #@param {type:"slider", min:10, max:200, step:10}

import os, glob, shutil, shlex
from google.colab import files
from IPython.display import Audio, display
from IPython import get_ipython

ip = get_ipython()
q = shlex.quote
os.chdir("/content/audio_project")

SVC_CKPT = os.path.abspath("checkpoints/YingMusic-SVC-full.pt")
EXP = f"exp_{RUN_ID}"
acc_arg = f"--accompany {q(ACCOMPANY)}" if ACCOMPANY else ""

print("🎤 กำลังประมวลผลแปลงเสียงร้อง...")
ip.system(
    f"python -u my_inference.py --source {q(SOURCE_VOCAL)} --target {q(TIMBRE_WAV)} "
    f"--diffusion-steps {int(DIFFUSION_STEPS)} --checkpoint {q(SVC_CKPT)} "
    f"--expname {q(EXP)} --cuda 0 --fp16 True {acc_arg} --config configs/YingMusic-SVC.yml"
)

vc_files = sorted(glob.glob(f"outputs/{EXP}/*.wav"))
mix_files = sorted(glob.glob(f"outputs/{EXP}/accompany/*.wav"))

if not vc_files:
    raise SystemExit("❌ ไม่พบไฟล์ผลลัพธ์ กรุณาตรวจสอบข้อความด้านบน")

results = [("เสียงร้องเดี่ยว (Vocal Only)", vc_files[-1], f"{SONG_TITLE}_vocal.wav")]
if mix_files:
    results.insert(0, ("เพลงเต็มที่มิกซ์แล้ว (Full Cover)", mix_files[-1], f"{SONG_TITLE}_cover.wav"))

for label, wav_path, out_name in results:
    final_wav = os.path.join(WORK, out_name)
    shutil.copy(wav_path, final_wav)
    preview_mp3 = os.path.splitext(final_wav)[0] + ".mp3"
    ip.system(f"ffmpeg -hide_banner -loglevel error -y -i {q(final_wav)} -b:a 192k {q(preview_mp3)}")
    print(f"\n✅ {label}:")
    display(Audio(filename=preview_mp3))
    files.download(final_wav)